In [65]:
from requests import request
from requests.compat import *
from bs4 import BeautifulSoup
import re
from requests.exceptions import HTTPError
from urllib.parse import urlparse
from urllib.parse import urljoin

headers = {
    'user-agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

# SBS 연예뉴스

In [20]:
k= int(input('몇 페이지 까지 검색하실 겁니까?:'))

for page in range(1,k+1):
    vurl ='https://ent.sbs.co.kr/news/flash.do?plink=GNB&cooper=SBSENTERNEWS&pageIdx={}'.format(page)
    urls = [vurl] # 앞으로 방문할곳
    visited = [] # 이미 방문한곳
    while urls: # 더이상 방문할 곳이 없을때까지
        url = urls.pop(0) # 앞에서부터 하나씩 꺼내고(FIFO)

        # HTTP 받아오고
        resp = request('GET', url, headers=headers)

        # 잘 받았으면

        visited.append(url)

            # Scarping(대상: text/html)
        if re.search('html', resp.headers['content-type']):
            # DOM 만들어주고
            dom = BeautifulSoup(resp.text, 'lxml')

            # 전략3-1. 특정 영역 - 뉴스 목록(주소만)
            # pagelink는 제외시킴
            news = dom.select('.news_list [href]')

            # 링크 찾기
            if len(news) > 0:
                a = []
                for temp in news:

                    temp = temp.attrs['href']
                    newurl = urljoin(url, temp)

                    a.append(newurl)


                    urls += list(filter(
                    lambda link:link not in visited and link not in urls, a))




            # 3-2. 뉴스 제목
            news_title = dom.select_one('.nwl_title')
            print(news_title)


            # 3-4. 뉴스 본문
            news_content = dom.select_one('.w_ctma_text')
            print(news_content)

            # 3-3 뉴스 날짜
            news_date = dom.select_one('.cth_text')
            print(news_date)            
            
            

#             if news:
#                 # https://ent.sbs.co.kr/news/article.do?article_id=E10010275065
#                 with open(url.split('/')[-1].replace('?', '.')+'.txt',
#                           'w', encoding='utf8') as fp:
#                     fp.write(news.text)
        # 못받았으면

            # 방문했다고 치고, 방문안하게
        else:
            visited.append(url)

몇 페이지 까지 검색하실 겁니까?:1


KeyError: 'origin-trial'

In [24]:
import pandas as pd
from requests import request
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

k = int(input('몇 페이지까지 검색하실 겁니까?:'))

title_list = []
content_list = []
date_list = []

for page in range(1, k + 1):
    vurl = 'https://ent.sbs.co.kr/news/flash.do?plink=GNB&cooper=SBSENTERNEWS&pageIdx={}'.format(page)
    urls = [vurl]  # 앞으로 방문할곳
    visited = []  # 이미 방문한곳
    while urls:  # 더이상 방문할 곳이 없을때까지
        url = urls.pop(0)  # 앞에서부터 하나씩 꺼내고(FIFO)

        # HTTP 받아오고
        resp = request('GET', url)

        # 잘 받았으면
        visited.append(url)

        # Scraping(대상: text/html)
        if re.search('html', resp.headers['content-type']):
            # DOM 만들어주고
            dom = BeautifulSoup(resp.text, 'lxml')

            # 전략3-1. 특정 영역 - 뉴스 목록(주소만)
            # pagelink는 제외시킴
            news = dom.select('.news_list [href]')

            # 링크 찾기
            if len(news) > 0:
                a = []
                for temp in news:
                    temp = temp.attrs['href']
                    newurl = urljoin(url, temp)
                    a.append(newurl)

                    urls += list(filter(lambda link: link not in visited and link not in urls, a))

            # 3-2. 뉴스 제목
            news_title = dom.select_one('.cth_title')
            if news_title:
                title_list.append(news_title.text.strip())

            # 3-4. 뉴스 본문
            news_content = dom.select_one('.w_ctma_text')
            if news_content:
                content_list.append(news_content.text.strip())

            # 3-3 뉴스 날짜
            news_date = dom.select_one('.cth_text')
            if news_date:
                date_list.append(news_date.text.strip())

pd.DataFrame({'날짜': date_list, '제목': title_list, '본문': content_list})

몇 페이지까지 검색하실 겁니까?:1


,날짜,제목,본문
0,작성 2023.08.08 16:06,"[스브수다] ""'시그널'시즌2, 존버해도 될까요?""…'악귀' 끝낸 김은희 작가에게 물었다",[SBS연예뉴스 | 강선애 기자] 김은희 작가는 자타공인 '장르물의 대가'다. '싸...
1,작성 2023.08.08 15:44,"'前국가대표' 김동성, 건설 노동자 변신 6개월째...""무더위에 끄덕없어""",[SBS 연예뉴스 ㅣ 강경윤 기자] 쇼트트랙 국가대표 출신 김동성이 건설 노동자로 ...
2,작성 2023.08.08 14:12,"연상호 감독, 영진위 지원 사업 폐지 반대 앞장…27인 공동 성명 발표",[SBS 연예뉴스 | 김지혜 기자] 연상호 감독을 비롯한 한국 장편 애니메이션 감독...
3,작성 2023.08.08 14:03,'엑소시스트' 윌리엄 프리드킨 감독 타계…유작은 베니스영화제서 공개,[SBS 연예뉴스 | 김지혜 기자] 공포 영화의 걸작 '엑소시스트'(1973)를 연...
4,작성 2023.08.08 13:21,"'콘크리트 유토피아', '밀수' 잡나…예매율 27.6%로 앞서",[SBS 연예뉴스 | 김지혜 기자] 영화 '콘크리트 유토피아'가 개봉을 하루 앞두고...
5,작성 2023.08.08 12:39,"""아내와 3m 거리 두는 게 좋아""…'돌싱포맨' 프로파일러 권일용, 아내에겐 약한 ...",[SBS연예뉴스 | 강선애 기자] 프로파일러 권일용이 정작 아내의 마음은 못 읽는다...
6,작성 2023.08.08 12:31,"추예진, 드라마 '두 남자' 캐스팅 확정…中 스타 호세군과 연기 호흡",[SBS연예뉴스 | 강선애 기자] 배우 추예진이 마운틴무브먼트 스토리가 제작하는 중...
7,작성 2023.08.08 11:41,"'몸값', 파라마운트+ 통해 글로벌 27개국 공개",[SBS 연예뉴스 | 김지혜 기자] 티빙 오리지널 시리즈 '몸값'이 오는 10월 파...
8,작성 2023.08.08 10:58,"덱스, 코로나19 확진...""해외 일정 중 감염, 일정 불참""",[SBS연예뉴스 | 강경윤 기자] 유튜버 겸 방송인 덱스(본명 김진영)가 코로나19...
9,작성 2023.08.08 10:50,"""얼렁뚱땅 잼버리, 수습을 왜 BTS가 해요?""...성일종 의원에 아미들 부글부글",[SBS연예뉴스 | 강경윤 기자] 국회 국방위원회 소속 성일종 국민의힘 의원이 미숙...


# 한국일보

In [68]:
k= int(input('몇 페이지 까지 검색하실 겁니까?:'))

title_list = []
content_list = []
date_list = []


for page in range(1,k+1):
    vurl ='https://www.hankookilbo.com/News/Culture/HF99&anchor'.format(page)
    urls = [vurl] # 앞으로 방문할곳
    visited = [] # 이미 방문한곳
    while urls: # 더이상 방문할 곳이 없을때까지
        url = urls.pop(0) # 앞에서부터 하나씩 꺼내고(FIFO)

        # HTTP 받아오고
        resp = request('GET', url, headers=headers)

        # 잘 받았으면

        visited.append(url)

            # Scarping(대상: text/html)
        if re.search('html', resp.headers['content-type']):
            # DOM 만들어주고
            dom = BeautifulSoup(resp.text, 'lxml')

            # 전략3-1. 특정 영역 - 뉴스 목록(주소만)
            # pagelink는 제외시킴
            news = dom.select('.wrap.imp-section > div > div > div.sub-contents > div.col-main > div > div.tab-contents.caladd [href]')

         #   body > div.wrap.imp-section > div > div > div.sub-contents > div.col-main > div > div.tab-contents.caladd
            
            # 링크 찾기
            if len(news) > 0:
                a = []
                for temp in news:

                    temp = temp.attrs['href']
                    newurl = urljoin(url, temp)

                    a.append(newurl)


                    urls += list(filter(
                    lambda link:link not in visited and link not in urls, a))


            # 3-2. 뉴스 제목
            news_title = dom.select_one('.title')
            if news_title:
                title_list.append(news_title.text.strip())

            # 3-3 뉴스 날짜
            news_date = dom.select_one('.wrt-text')
            print(news_date)
            
            # 3-4 뉴스 기자
            news_reporter = dom.select_one('.name')
            print(news_date) 

              # 3-4. 뉴스 본문
            news_content = dom.select_one('.col-main')
            print(news_content)
            
            
            if news:
                 # https://ent.sbs.co.kr/news/article.do?article_id=E10010275065
                with open(url.split('/')[-1].replace('?', '.')+'.txt',
                           'w', encoding='utf8') as fp:
                     fp.write(news.text)


            # 방문했다고 치고, 방문안하게
        else:
            visited.append(url)

pd.DataFrame({'날짜': date_list, '제목': title_list, '본문': content_list})

몇 페이지 까지 검색하실 겁니까?:1


,날짜,제목,본문


In [48]:
news = dom.select('.div.tab-contents.caladd')
print(news)

[]
